# 04 — Model Selection

## Objective

Compare several machine learning algorithms for credit-card fraud detection
using the same preprocessing and evaluation methodology.

The goals of this notebook are to:

- Compare baseline candidate algorithms
- Evaluate performance on the untouched test set
- Use cross-validation to assess model consistency
- Identify models suitable for hyperparameter tuning

## Important methodology

The test set remains completely untouched during model selection.

SMOTE is used for model training to address severe class imbalance.

Cross-validation is performed separately on the original training data
to avoid evaluating on a dataset that was SMOTE-balanced before the folds
were created.

Hyperparameter tuning and threshold optimization are handled in the
next notebook.

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)

from xgboost import XGBClassifier

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    cross_val_score
)

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

In [2]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

In [3]:
X_train_original = pd.read_csv(
    "../data/Processed/X_train_processed.csv"
)

X_test = pd.read_csv(
    "../data/Processed/X_test_processed.csv"
)

In [4]:
df = pd.read_csv(
    "../data/Processed/creditcard_clean.csv"
)

In [5]:
from sklearn.model_selection import train_test_split

X_full = df.drop(columns=["Class"])
y_full = df["Class"]

X_train_full, X_test_full, y_train_original, y_test_original = train_test_split(
    X_full,
    y_full,
    test_size=0.20,
    stratify=y_full,
    random_state=42
)

In [6]:
y_test = pd.read_csv(
    "../data/Processed/y_test.csv"
).squeeze()

In [7]:
print("Original training target:")
print(y_train_original.value_counts())

print("\nTest target:")
print(y_test.value_counts())

Original training target:
Class
0    226602
1       378
Name: count, dtype: int64

Test target:
Class
0    56651
1       95
Name: count, dtype: int64


In [8]:
print("X_train_original:", X_train_original.shape)
print("X_test:", X_test.shape)

print("\nX_train_original data types:")
print(X_train_original.dtypes)

X_train_original: (226980, 32)
X_test: (56746, 32)

X_train_original data types:
Amount      float64
Time        float64
V1          float64
V2          float64
V3          float64
V4          float64
V5          float64
V6          float64
V7          float64
V8          float64
V9          float64
V10         float64
V11         float64
V12         float64
V13         float64
V14         float64
V15         float64
V16         float64
V17         float64
V18         float64
V19         float64
V20         float64
V21         float64
V22         float64
V23         float64
V24         float64
V25         float64
V26         float64
V27         float64
V28         float64
Hour_sin    float64
Hour_cos    float64
dtype: object


## 4. SMOTE Training Data

SMOTE is applied only to the training data.

The test set remains untouched and retains its original class distribution.

This produces a balanced training dataset for the main model-selection
experiment.

In [9]:
smote = SMOTE(
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_original,
    y_train_original
)

In [10]:
print("X_train_smote:", X_train_smote.shape)
print("y_train_smote:", y_train_smote.shape)

print("\nSMOTE class distribution:")
print(y_train_smote.value_counts())

X_train_smote: (453204, 32)
y_train_smote: (453204,)

SMOTE class distribution:
Class
0    226602
1    226602
Name: count, dtype: int64


## 5. Evaluation Function

The evaluation function calculates:

- Precision
- Recall
- F1-score
- ROC-AUC
- PR-AUC
- Confusion matrix components

PR-AUC is particularly important because fraud is highly imbalanced.

In [11]:
def evaluate_model(model, X_test, y_test):

    # Class predictions
    y_pred = model.predict(X_test)

    # Probability of class 1 (fraud)
    y_prob = model.predict_proba(X_test)[:, 1]

    # Classification metrics
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # Ranking metrics
    roc_auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()

    return {
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

## 6. Candidate Models

The following algorithms will be compared:

1. Logistic Regression
2. Random Forest
3. Extra Trees
4. XGBoost
5. HistGradientBoosting

The models use baseline configurations only.

Hyperparameter optimization is intentionally excluded from this notebook.

In [12]:
models = {

    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=100,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    ),

    "HistGradientBoosting": HistGradientBoostingClassifier(
        random_state=42
    )
}

## 7. Test-Set Model Comparison

Each model is trained on the SMOTE-balanced training data.

The test set is completely untouched.

This allows direct comparison with the baseline models from
`03_baseline.ipynb`.

In [13]:
results = []

for name, model in models.items():

    print(f"Training {name}...")

    # Train on SMOTE-balanced training data
    model.fit(
        X_train_smote,
        y_train_smote
    )

    # Evaluate on untouched test data
    metrics = evaluate_model(
        model,
        X_test,
        y_test
    )

    metrics["Model"] = name

    results.append(metrics)

    print(f"{name} complete.\n")

Training Logistic Regression...
Logistic Regression complete.

Training Random Forest...
Random Forest complete.

Training Extra Trees...
Extra Trees complete.

Training XGBoost...
XGBoost complete.

Training HistGradientBoosting...
HistGradientBoosting complete.



In [14]:
results_df = pd.DataFrame(results)

results_df = results_df[
    [
        "Model",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "PR-AUC",
        "TN",
        "FP",
        "FN",
        "TP"
    ]
]

In [15]:
results_display = results_df.copy()

metric_cols = [
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC",
    "PR-AUC"
]

results_display[metric_cols] = (
    results_display[metric_cols] * 100
).round(2)

results_display

,Model,Precision,Recall,F1,ROC-AUC,PR-AUC,TN,FP,FN,TP
0,Logistic Regression,5.25,87.37,9.90,96.38,68.67,55153,1498,12,83
1,Random Forest,92.50,77.89,84.57,96.47,81.84,56645,6,21,74
2,Extra Trees,93.51,75.79,83.72,95.90,81.50,56646,5,23,72
3,XGBoost,88.10,77.89,82.68,96.70,81.41,56641,10,21,74
4,HistGradientBoosting,79.35,76.84,78.07,96.07,79.73,56632,19,22,73


## 8. Cross-Validation

Cross-validation is performed on the original training data.

SMOTE is NOT applied before creating the folds.

This provides a clean reference for model consistency without allowing
synthetic observations to influence the validation folds.

The final test set remains untouched.

In [16]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [17]:
scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

In [18]:
cv_models = {

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=100,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    )
}

In [19]:
cv_results = []

for name, model in cv_models.items():

    print(f"Running CV for {name}...")

    scores = cross_validate(
        model,
        X_train_original,
        y_train_original,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    cv_results.append({

        "Model": name,

        "Precision Mean":
            scores["test_precision"].mean(),

        "Precision Std":
            scores["test_precision"].std(),

        "Recall Mean":
            scores["test_recall"].mean(),

        "Recall Std":
            scores["test_recall"].std(),

        "F1 Mean":
            scores["test_f1"].mean(),

        "F1 Std":
            scores["test_f1"].std(),

        "ROC-AUC Mean":
            scores["test_roc_auc"].mean(),

        "ROC-AUC Std":
            scores["test_roc_auc"].std(),

        "PR-AUC Mean":
            scores["test_pr_auc"].mean(),

        "PR-AUC Std":
            scores["test_pr_auc"].std()
    })

    print(f"{name} complete.\n")

Running CV for Random Forest...
Random Forest complete.

Running CV for Extra Trees...
Extra Trees complete.

Running CV for XGBoost...
XGBoost complete.



In [20]:
cv_results_df = pd.DataFrame(cv_results)

cv_results_df

,Model,Precision Mean,Precision Std,Recall Mean,Recall Std,F1 Mean,F1 Std,ROC-AUC Mean,ROC-AUC Std,PR-AUC Mean,PR-AUC Std
0,Random Forest,0.937427,0.024645,0.772421,0.080278,0.844187,0.047385,0.940120,0.015532,0.831225,0.028977
1,Extra Trees,0.948705,0.019121,0.761930,0.063845,0.843123,0.037635,0.942470,0.006876,0.839668,0.027943
2,XGBoost,0.691620,0.336812,0.612561,0.295907,0.648247,0.312971,0.847662,0.179092,0.606152,0.311926


## 9. Model Comparison

The test-set results represent performance after training on SMOTE-balanced
data and evaluating against the untouched test distribution.

The cross-validation results provide an additional assessment of model
consistency using the original training distribution.

The two evaluations serve different purposes and should not be interpreted
as identical estimates.

In [21]:
results_display

,Model,Precision,Recall,F1,ROC-AUC,PR-AUC,TN,FP,FN,TP
0,Logistic Regression,5.25,87.37,9.90,96.38,68.67,55153,1498,12,83
1,Random Forest,92.50,77.89,84.57,96.47,81.84,56645,6,21,74
2,Extra Trees,93.51,75.79,83.72,95.90,81.50,56646,5,23,72
3,XGBoost,88.10,77.89,82.68,96.70,81.41,56641,10,21,74
4,HistGradientBoosting,79.35,76.84,78.07,96.07,79.73,56632,19,22,73


## 9. Compare Test-Set and Cross-Validation Results

The test-set results show how each model performed on the untouched test set.

The cross-validation results show how consistently each model performed
across different folds of the original training data.

Comparing the two helps identify:

- Models whose performance is relatively consistent
- Models with substantial variation across folds
- Differences between validation performance and final test performance

PR-AUC is given particular attention because the dataset is highly
imbalanced and PR-AUC focuses on the precision-recall trade-off for the
minority class.

In [22]:
test_comparison = results_df[
    ["Model", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]
].copy()

test_comparison

,Model,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Logistic Regression,0.052498,0.873684,0.099045,0.963780,0.686703
1,Random Forest,0.925000,0.778947,0.845714,0.964746,0.818448
2,Extra Trees,0.935065,0.757895,0.837209,0.958961,0.814994
3,XGBoost,0.880952,0.778947,0.826816,0.967021,0.814062
4,HistGradientBoosting,0.793478,0.768421,0.780749,0.960653,0.797262


In [23]:
cv_comparison = cv_results_df[
    [
        "Model",
        "Precision Mean",
        "Precision Std",
        "Recall Mean",
        "Recall Std",
        "F1 Mean",
        "F1 Std",
        "ROC-AUC Mean",
        "ROC-AUC Std",
        "PR-AUC Mean",
        "PR-AUC Std"
    ]
].copy()

cv_comparison

,Model,Precision Mean,Precision Std,Recall Mean,Recall Std,F1 Mean,F1 Std,ROC-AUC Mean,ROC-AUC Std,PR-AUC Mean,PR-AUC Std
0,Random Forest,0.937427,0.024645,0.772421,0.080278,0.844187,0.047385,0.940120,0.015532,0.831225,0.028977
1,Extra Trees,0.948705,0.019121,0.761930,0.063845,0.843123,0.037635,0.942470,0.006876,0.839668,0.027943
2,XGBoost,0.691620,0.336812,0.612561,0.295907,0.648247,0.312971,0.847662,0.179092,0.606152,0.311926


In [24]:
pr_auc_comparison = cv_results_df[
    ["Model", "PR-AUC Mean", "PR-AUC Std"]
].copy()

pr_auc_comparison["PR-AUC Range"] = (
    pr_auc_comparison["PR-AUC Mean"]
    .astype(str)
    + " ± "
    + pr_auc_comparison["PR-AUC Std"].round(4).astype(str)
)

pr_auc_comparison[
    ["Model", "PR-AUC Range"]
]

,Model,PR-AUC Range
0,Random Forest,0.8312248249401092 ± 0.029
1,Extra Trees,0.8396682542820054 ± 0.0279
2,XGBoost,0.6061521663680233 ± 0.3119


In [25]:
cv_results_df

,Model,Precision Mean,Precision Std,Recall Mean,Recall Std,F1 Mean,F1 Std,ROC-AUC Mean,ROC-AUC Std,PR-AUC Mean,PR-AUC Std
0,Random Forest,0.937427,0.024645,0.772421,0.080278,0.844187,0.047385,0.940120,0.015532,0.831225,0.028977
1,Extra Trees,0.948705,0.019121,0.761930,0.063845,0.843123,0.037635,0.942470,0.006876,0.839668,0.027943
2,XGBoost,0.691620,0.336812,0.612561,0.295907,0.648247,0.312971,0.847662,0.179092,0.606152,0.311926


### Interpretation

The test-set and cross-validation results provide complementary information.

Random Forest shows relatively consistent performance across the CV folds,
with a PR-AUC mean of approximately 0.83 and a relatively small standard
deviation.

Extra Trees also shows consistent CV performance, with a PR-AUC mean of
approximately 0.84 and a similarly small standard deviation.

XGBoost shows substantially greater variation across the CV folds. Its
test-set PR-AUC is considerably higher than its CV mean, indicating that its
performance is less consistent under the current CV setup.

These results justify carrying Random Forest, Extra Trees, and XGBoost
forward to the model-tuning stage rather than eliminating a model based on
a single metric alone.

The next notebook will evaluate these candidates using hyperparameter
tuning with SMOTE applied within each cross-validation training fold.

## 10. Save Model Selection Results

The model-selection results are saved as artifacts for later reference.

The model-selection results are saved as CSV files in the artifacts
directory.

These files contain:

- Test-set performance for all candidate models
- Cross-validation performance for the selected candidates

The results will be used as a reference during hyperparameter tuning.

In [26]:
results_df.to_csv(
    "../artifacts/model_selection_test_results.csv",
    index=False
)

cv_results_df.to_csv(
    "../artifacts/model_selection_cv_results.csv",
    index=False
)

print("Model selection results saved.")

Model selection results saved.


In [28]:
import os

print(
    "Test results:",
    os.path.exists("../artifacts/model_selection_test_results.csv")
)

print(
    "CV results:",
    os.path.exists("../artifacts/model_selection_cv_results.csv")
)

Test results: True
CV results: True


## 11. Model Selection Conclusion

The candidate models were evaluated using consistent preprocessing and an
untouched test set.

Random Forest, Extra Trees, and XGBoost were selected for further analysis
because they demonstrated competitive performance across the primary fraud
detection metrics.

Cross-validation was used as a secondary assessment of model consistency.

The selected models will be investigated further in
`05_model_tuning.ipynb`.

Hyperparameter tuning, threshold optimization, and final model selection
are intentionally deferred to the next stage.